In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

In [2]:
from catboost import CatBoostRegressor

TRAIN_PATH = r"C:\Users\Gusti Jogish\Downloads\Gammafest\dataset\train.csv"
TEST_PATH = r"C:\Users\Gusti Jogish\Downloads\Gammafest\dataset\test.csv"
SAMPLE_SUB_PATH = r"C:\Users\Gusti Jogish\Downloads\Gammafest\dataset\sample submission.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Sample shape:", sample_sub.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

Train shape: (78772, 47)
Test shape : (42422, 20)
Sample shape: (42422, 3)

Train columns:
['Id', 'match_id', 'date', 'gender', 'team', 'opponent', 'is_home', 'neutral', 'tournament', 'venue_country', 'team_goals', 'opp_goals', 'team_points_last5', 'opp_points_last5', 'points_last5_diff', 'team_gd_last5', 'opp_gd_last5', 'gd_last5_diff', 'h2h_points_last5', 'h2h_gd_last5', 'days_since_last_match_team', 'days_since_last_match_opp', 'team_points_last10', 'opp_points_last10', 'team_avg_goals_last5', 'team_avg_conceded_last5', 'opp_avg_goals_last5', 'opp_avg_conceded_last5', 'team_win_rate_last10', 'opp_win_rate_last10', 'elo_team', 'elo_opponent', 'rank_team', 'rank_opponent', 'rank_diff', 'rank_missing_team', 'rank_missing_opp', 'confederation_team', 'confederation_opp', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue']

Test columns:
['Id', 'match_id', 'date', 'gender', 

Data Understanading

In [3]:
missing_train = pd.DataFrame({
    "train_missing_count": train.isna().sum(),
    "train_missing_pct": (train.isna().sum() / len(train)) * 100
})

missing_test = pd.DataFrame({
    "test_missing_count": test.isna().sum(),
    "test_missing_pct": (test.isna().sum() / len(test)) * 100
})

missing_summary = missing_train.join(missing_test, how="outer").fillna(0)
missing_summary = missing_summary.sort_values("test_missing_pct", ascending=False)

display(missing_summary)

,train_missing_count,train_missing_pct,test_missing_count,test_missing_pct
distance_travel_opp,30392,38.582237,16975.0,40.014615
distance_travel_team,30392,38.582237,16975.0,40.014615
gdp_per_capita_team,25957,32.952064,14664.0,34.566970
gdp_per_capita_opp,25957,32.952064,14664.0,34.566970
altitude_venue,20554,26.093028,10906.0,25.708359
temperature_venue,17074,21.675215,5594.0,13.186554
population_team,14872,18.879805,3372.0,7.948706
population_opp,14872,18.879805,3372.0,7.948706
days_since_last_match_team,294,0.373229,0.0,0.000000
date,0,0.000000,0.0,0.000000


In [4]:
if "team_model" not in globals() or "final_features" not in globals():
    print("Run training cells first (team_model/final_features belum ada).")
else:
    fi_team = pd.DataFrame({
        "feature": final_features,
        "importance": team_model.get_feature_importance()
    }).sort_values("importance", ascending=False)
    display(fi_team.head(25))


Run training cells first (team_model/final_features belum ada).


In [5]:
if "opp_model" not in globals() or "final_features" not in globals():
    print("Run training cells first (opp_model/final_features belum ada).")
else:
    fi_opp = pd.DataFrame({
        "feature": final_features,
        "importance": opp_model.get_feature_importance()
    }).sort_values("importance", ascending=False)
    display(fi_opp.head(25))


Run training cells first (opp_model/final_features belum ada).


In [6]:
if "fi_team" not in globals() or "fi_opp" not in globals():
    print("Run cell feature importance team dan opp dulu.")
else:
    fi_merge = fi_team.merge(fi_opp, on="feature", suffixes=("_team", "_opp"))
    fi_merge["importance_avg"] = (fi_merge["importance_team"] + fi_merge["importance_opp"]) / 2
    fi_merge = fi_merge.sort_values("importance_avg", ascending=False)
    display(fi_merge.head(30))


Run cell feature importance team dan opp dulu.


In [7]:
selected_features_ramping = [
    "opponent",
    "team",
    "gender",
    "population_opp",
    "population_team",
    "is_home",
    "tournament",
    "confederation_team",
    "confederation_opp",
    "gdp_per_capita_opp",
    "gdp_per_capita_team",
    "venue_country",
    "distance_travel_team",
    "distance_travel_opp",
    "altitude_venue",
    "temperature_venue",
    "day",
    "month",
    "year"
]

print("Jumlah kandidat feature ramping:", len(selected_features_ramping))
print(selected_features_ramping)


Jumlah kandidat feature ramping: 19
['opponent', 'team', 'gender', 'population_opp', 'population_team', 'is_home', 'tournament', 'confederation_team', 'confederation_opp', 'gdp_per_capita_opp', 'gdp_per_capita_team', 'venue_country', 'distance_travel_team', 'distance_travel_opp', 'altitude_venue', 'temperature_venue', 'day', 'month', 'year']


Common Feature

In [8]:
train_cols = set(train.columns)
test_cols = set(test.columns)

common_cols = sorted(list(train_cols.intersection(test_cols)))
train_only_cols = sorted(list(train_cols - test_cols))
test_only_cols = sorted(list(test_cols - train_cols))

print("Jumlah common columns :", len(common_cols))
print("Jumlah train-only cols:", len(train_only_cols))
print("Jumlah test-only cols :", len(test_only_cols))

print("\nCommon columns:")
print(common_cols)

print("\nTrain-only columns:")
print(train_only_cols)

Jumlah common columns : 20
Jumlah train-only cols: 27
Jumlah test-only cols : 0

Common columns:
['Id', 'altitude_venue', 'confederation_opp', 'confederation_team', 'date', 'distance_travel_opp', 'distance_travel_team', 'gdp_per_capita_opp', 'gdp_per_capita_team', 'gender', 'is_home', 'match_id', 'neutral', 'opponent', 'population_opp', 'population_team', 'team', 'temperature_venue', 'tournament', 'venue_country']

Train-only columns:
['days_since_last_match_opp', 'days_since_last_match_team', 'elo_opponent', 'elo_team', 'gd_last5_diff', 'h2h_gd_last5', 'h2h_points_last5', 'opp_avg_conceded_last5', 'opp_avg_goals_last5', 'opp_gd_last5', 'opp_goals', 'opp_points_last10', 'opp_points_last5', 'opp_win_rate_last10', 'points_last5_diff', 'rank_diff', 'rank_missing_opp', 'rank_missing_team', 'rank_opponent', 'rank_team', 'team_avg_conceded_last5', 'team_avg_goals_last5', 'team_gd_last5', 'team_goals', 'team_points_last10', 'team_points_last5', 'team_win_rate_last10']


AW MAE Baseline

In [9]:
TOURNAMENT_WEIGHTS = {
    "FIFA World Cup": 2.00,
    "AFC Asian Cup": 1.80,
    "African Cup of Nations": 1.80,
    "UEFA Euro": 1.90,
    "Copa América": 1.90,
    "Confederations Cup": 1.70,
    "Olympic Games": 1.50,
    "Friendly": 0.96
}

DEFAULT_WEIGHT = 1.20

def get_match_outcome(team_goals, opp_goals):
    if team_goals > opp_goals:
        return 1
    elif team_goals < opp_goals:
        return -1
    return 0

def compute_match_loss(y_true_team, y_true_opp, y_pred_team, y_pred_opp, tournament_name):
    # pastikan integer non-negatif
    y_pred_team = max(0, int(round(y_pred_team)))
    y_pred_opp = max(0, int(round(y_pred_opp)))

    # 1. Base MAE
    mae = (abs(y_true_team - y_pred_team) + abs(y_true_opp - y_pred_opp)) / 2

    # 2. Penalty
    exact = int((y_true_team == y_pred_team) and (y_true_opp == y_pred_opp))
    outcome = int(get_match_outcome(y_true_team, y_true_opp) == get_match_outcome(y_pred_team, y_pred_opp))
    gd = int((y_true_team - y_true_opp) == (y_pred_team - y_pred_opp))

    penalty = 0.30 * (1 - exact) + 0.25 * (1 - outcome) + 0.15 * (1 - gd)

    # 3. Outcome multiplier
    multiplier = 1.0 if outcome == 1 else 1.5

    # 4. Non-linear scaling
    raw_loss = mae + penalty
    loss = (raw_loss * multiplier) ** 1.5

    # 5. Tournament weighting
    weight = TOURNAMENT_WEIGHTS.get(tournament_name, DEFAULT_WEIGHT)

    return loss, weight

def aw_mae_score(df_eval, true_team_col, true_opp_col, pred_team_col, pred_opp_col, tournament_col="tournament"):
    weighted_losses = []
    weights = []

    for _, row in df_eval.iterrows():
        loss, weight = compute_match_loss(
            row[true_team_col],
            row[true_opp_col],
            row[pred_team_col],
            row[pred_opp_col],
            row[tournament_col]
        )
        weighted_losses.append(loss * weight)
        weights.append(weight)

    return np.sum(weighted_losses) / np.sum(weights)

Prepro

In [10]:
train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

for df in [train, test]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["decade"] = (df["year"] // 10) * 10

print(train[["date", "year", "month", "dayofweek", "decade"]].head())

        date  year  month  dayofweek  decade
0 1872-11-30  1872     11          5    1870
1 1872-11-30  1872     11          5    1870
2 1873-03-08  1873      3          5    1870
3 1873-03-08  1873      3          5    1870
4 1874-03-07  1874      3          5    1870


In [11]:
drop_cols = ["Id", "match_id", "date", "team_goals", "opp_goals"]

base_features = [col for col in test.columns if col not in ["Id", "match_id", "date"]]
base_features += ["year", "month", "day", "dayofweek", "is_weekend", "decade"]

# pastikan cuma ambil yang ada di train juga
base_features = [col for col in base_features if col in train.columns]

print("Jumlah feature baseline:", len(base_features))
print(base_features)

categorical_features = [
    col for col in base_features
    if train[col].dtype == "object"
]

numerical_features = [
    col for col in base_features
    if col not in categorical_features
]

print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Jumlah feature baseline: 29
['gender', 'team', 'opponent', 'is_home', 'neutral', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade']

Categorical features:
['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp']

Numerical features:
['is_home', 'neutral', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade']


In [12]:
exclude_cols = ["Id", "match_id", "date", "team_goals", "opp_goals"]

base_features = [col for col in train.columns if col in test.columns and col not in exclude_cols]

# tambahkan fitur turunan tanggal kalau memang belum ada
extra_date_features = ["year", "month", "day", "dayofweek", "is_weekend", "decade"]
for col in extra_date_features:
    if col in train.columns and col in test.columns and col not in base_features:
        base_features.append(col)

# pastikan unique, urutan tetap aman
base_features = list(dict.fromkeys(base_features))

print("Jumlah feature baseline:", len(base_features))
print(base_features)

categorical_features = [col for col in base_features if train[col].dtype == "object"]
numerical_features = [col for col in base_features if col not in categorical_features]

print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

# cek duplikat nama kolom
dup_cols = pd.Series(base_features)[pd.Series(base_features).duplicated()].tolist()
print("\nDuplicate feature names:", dup_cols)

Jumlah feature baseline: 23
['gender', 'team', 'opponent', 'is_home', 'neutral', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade']

Categorical features:
['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp']

Numerical features:
['is_home', 'neutral', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue', 'year', 'month', 'day', 'dayofweek', 'is_weekend', 'decade']

Duplicate feature names: []


In [13]:
MODERN_YEAR = 1990

train_modern = train[train["year"] >= MODERN_YEAR].copy().reset_index(drop=True)

print("Original train shape :", train.shape)
print("Modern train shape   :", train_modern.shape)
print("Year min-max modern  :", train_modern["year"].min(), "-", train_modern["year"].max())


Original train shape : (78772, 53)
Modern train shape   : (43610, 53)
Year min-max modern  : 1990 - 2011


In [14]:
exclude_cols = ["Id", "match_id", "date", "team_goals", "opp_goals"]

reference_features = selected_features_ramping if "selected_features_ramping" in globals() else base_features

keep_cols = []
impute_cols = []
drop_cols_missing = []

for col in reference_features:
    if col not in train.columns or col not in test.columns:
        continue

    test_missing_pct = missing_summary.loc[col, "test_missing_pct"] if col in missing_summary.index else 0

    if test_missing_pct > 50:
        drop_cols_missing.append(col)
    elif test_missing_pct > 0:
        impute_cols.append(col)
    else:
        keep_cols.append(col)

print("=== KEEP COLS ===")
print(keep_cols)

print("\n=== IMPUTE COLS ===")
print(impute_cols)

print("\n=== DROP COLS (missing test > 50%) ===")
print(drop_cols_missing)


=== KEEP COLS ===
['opponent', 'team', 'gender', 'is_home', 'tournament', 'confederation_team', 'confederation_opp', 'venue_country', 'day', 'month', 'year']

=== IMPUTE COLS ===
['population_opp', 'population_team', 'gdp_per_capita_opp', 'gdp_per_capita_team', 'distance_travel_team', 'distance_travel_opp', 'altitude_venue', 'temperature_venue']

=== DROP COLS (missing test > 50%) ===
[]


In [15]:
reference_features = selected_features_ramping if "selected_features_ramping" in globals() else base_features

final_features = [
    col for col in reference_features
    if col in train_modern.columns and col in test.columns and col not in drop_cols_missing
]

categorical_features = [
    col for col in final_features
    if train_modern[col].dtype == "object"
]

numerical_features = [
    col for col in final_features
    if col not in categorical_features
]

print("Jumlah final features:", len(final_features))
print(final_features)

print("\nCategorical:")
print(categorical_features)

print("\nNumerical:")
print(numerical_features)


Jumlah final features: 19
['opponent', 'team', 'gender', 'population_opp', 'population_team', 'is_home', 'tournament', 'confederation_team', 'confederation_opp', 'gdp_per_capita_opp', 'gdp_per_capita_team', 'venue_country', 'distance_travel_team', 'distance_travel_opp', 'altitude_venue', 'temperature_venue', 'day', 'month', 'year']

Categorical:
['opponent', 'team', 'gender', 'tournament', 'confederation_team', 'confederation_opp', 'venue_country']

Numerical:
['population_opp', 'population_team', 'is_home', 'gdp_per_capita_opp', 'gdp_per_capita_team', 'distance_travel_team', 'distance_travel_opp', 'altitude_venue', 'temperature_venue', 'day', 'month', 'year']


In [16]:
train_modern_imp = train_modern.copy()
test_imp = test.copy()

for col in categorical_features:
    train_modern_imp[col] = train_modern_imp[col].fillna("Unknown")
    test_imp[col] = test_imp[col].fillna("Unknown")

for col in numerical_features:
    med = train_modern_imp[col].median()
    train_modern_imp[col] = train_modern_imp[col].fillna(med)
    test_imp[col] = test_imp[col].fillna(med)

print("Imputasi selesai.")
print("Train missing remaining:", train_modern_imp[final_features].isna().sum().sum())
print("Test missing remaining:", test_imp[final_features].isna().sum().sum())


Imputasi selesai.
Train missing remaining: 0
Test missing remaining: 0


Split

In [17]:
train_modern_imp = train_modern_imp.sort_values("date").reset_index(drop=True)

split_date = train_modern_imp["date"].quantile(0.85)

train_part = train_modern_imp[train_modern_imp["date"] < split_date].copy()
valid_part = train_modern_imp[train_modern_imp["date"] >= split_date].copy()

X_train = train_part[final_features].copy()
X_valid = valid_part[final_features].copy()

y_train_team = train_part["team_goals"].copy()
y_train_opp = train_part["opp_goals"].copy()

y_valid_team = valid_part["team_goals"].copy()
y_valid_opp = valid_part["opp_goals"].copy()

print("Train shape:", X_train.shape)
print("Valid shape:", X_valid.shape)

Train shape: (37066, 19)
Valid shape: (6544, 19)


training `team_goals`

In [18]:
team_model = CatBoostRegressor(
    iterations=700,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    verbose=100
)

team_model.fit(
    X_train,
    y_train_team,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid_team),
    use_best_model=True
)

0:	learn: 1.1582561	test: 1.1118956	best: 1.1118956 (0)	total: 117ms	remaining: 1m 21s
100:	learn: 1.0377527	test: 1.0198262	best: 1.0198225 (99)	total: 3.57s	remaining: 21.2s
200:	learn: 1.0183571	test: 1.0161607	best: 1.0160154 (194)	total: 7.1s	remaining: 17.6s
300:	learn: 1.0036249	test: 1.0133506	best: 1.0133506 (300)	total: 10.6s	remaining: 14.1s
400:	learn: 0.9902469	test: 1.0098853	best: 1.0098853 (400)	total: 14.3s	remaining: 10.6s
500:	learn: 0.9805370	test: 1.0083577	best: 1.0083284 (499)	total: 17.7s	remaining: 7.04s
600:	learn: 0.9728066	test: 1.0084329	best: 1.0080438 (521)	total: 21.2s	remaining: 3.49s
699:	learn: 0.9659425	test: 1.0081499	best: 1.0080438 (521)	total: 24.7s	remaining: 0us

bestTest = 1.008043796
bestIteration = 521

Shrink model to first 522 iterations.


training `opp_goals`

In [19]:
opp_model = CatBoostRegressor(
    iterations=700,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    verbose=100
)

opp_model.fit(
    X_train,
    y_train_opp,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid_opp),
    use_best_model=True
)

0:	learn: 1.1579378	test: 1.1102682	best: 1.1102682 (0)	total: 38.6ms	remaining: 27s
100:	learn: 1.0385187	test: 1.0228640	best: 1.0228640 (100)	total: 3.46s	remaining: 20.5s
200:	learn: 1.0172079	test: 1.0174326	best: 1.0174326 (200)	total: 6.92s	remaining: 17.2s
300:	learn: 1.0018531	test: 1.0133876	best: 1.0133521 (295)	total: 10.4s	remaining: 13.8s
400:	learn: 0.9896760	test: 1.0103272	best: 1.0103272 (400)	total: 14s	remaining: 10.4s
500:	learn: 0.9796880	test: 1.0097958	best: 1.0097545 (489)	total: 17.5s	remaining: 6.96s
600:	learn: 0.9716365	test: 1.0089999	best: 1.0088877 (563)	total: 21.1s	remaining: 3.48s
699:	learn: 0.9642501	test: 1.0086881	best: 1.0086832 (692)	total: 24.6s	remaining: 0us

bestTest = 1.008683218
bestIteration = 692

Shrink model to first 693 iterations.


eval aw-mae

In [20]:
valid_pred_team = team_model.predict(X_valid)
valid_pred_opp = opp_model.predict(X_valid)

print("Raw prediction stats:")
print("team -> min:", np.min(valid_pred_team), "max:", np.max(valid_pred_team), "mean:", np.mean(valid_pred_team))
print("opp  -> min:", np.min(valid_pred_opp), "max:", np.max(valid_pred_opp), "mean:", np.mean(valid_pred_opp))

valid_eval = valid_part.copy()
valid_eval["pred_team_goals"] = np.round(np.clip(valid_pred_team, 0, 5)).astype(int)
valid_eval["pred_opp_goals"] = np.round(np.clip(valid_pred_opp, 0, 5)).astype(int)

awmae = aw_mae_score(
    valid_eval,
    true_team_col="team_goals",
    true_opp_col="opp_goals",
    pred_team_col="pred_team_goals",
    pred_opp_col="pred_opp_goals",
    tournament_col="tournament"
)

mae_team = mean_absolute_error(valid_eval["team_goals"], valid_eval["pred_team_goals"])
mae_opp = mean_absolute_error(valid_eval["opp_goals"], valid_eval["pred_opp_goals"])
overall_mae = (
    (valid_eval["team_goals"] - valid_eval["pred_team_goals"]).abs() +
    (valid_eval["opp_goals"] - valid_eval["pred_opp_goals"]).abs()
).mean() / 2

print("\nValidation MAE team :", mae_team)
print("Validation MAE opp  :", mae_opp)
print("Validation Overall MAE:", overall_mae)
print("Validation AW-MAE   :", awmae)

display(valid_eval[[
    "date", "team", "opponent", "tournament",
    "team_goals", "opp_goals",
    "pred_team_goals", "pred_opp_goals"
]].head(20))

Raw prediction stats:
team -> min: -0.6268733448955486 max: 7.843425647148314 mean: 1.2532732681463352
opp  -> min: -0.5728245052928804 max: 7.812484284872254 mean: 1.2685118009597292

Validation MAE team : 0.9847188264058679
Validation MAE opp  : 0.9844132029339854
Validation Overall MAE: 0.9845660146699267
Validation AW-MAE   : 3.116338852448076


,date,team,opponent,tournament,team_goals,opp_goals,pred_team_goals,pred_opp_goals
37066,2008-10-20,Australia,Vietnam,AFF Championship,1,0,2,1
37067,2008-10-20,Vietnam,Australia,AFF Championship,0,1,1,2
37068,2008-10-20,Myanmar,Thailand,AFF Championship,0,3,1,3
37069,2008-10-20,Afghanistan,Malaysia,Merdeka Tournament,0,6,0,3
37070,2008-10-20,Malaysia,Afghanistan,Merdeka Tournament,6,0,3,0
37071,2008-10-20,Thailand,Myanmar,AFF Championship,3,0,3,1
37072,2008-10-21,Timor-Leste,Brunei,AFF Championship qualification,1,4,2,1
37073,2008-10-21,Brunei,Timor-Leste,AFF Championship qualification,4,1,2,2
37074,2008-10-21,Philippines,Laos,AFF Championship qualification,1,2,2,1
37075,2008-10-21,Laos,Philippines,AFF Championship qualification,2,1,1,2


-----------------

train full model buat submission

In [21]:
X_full = train_modern_imp[final_features].copy()
y_full_team = train_modern_imp["team_goals"].copy()
y_full_opp = train_modern_imp["opp_goals"].copy()

X_test = test_imp[final_features].copy()

best_iter_team = team_model.get_best_iteration()
best_iter_opp = opp_model.get_best_iteration()

final_team_model = CatBoostRegressor(
    iterations=best_iter_team if best_iter_team is not None and best_iter_team > 0 else 700,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    random_seed=42,
    verbose=100
)

final_opp_model = CatBoostRegressor(
    iterations=best_iter_opp if best_iter_opp is not None and best_iter_opp > 0 else 700,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    random_seed=42,
    verbose=100
)

final_team_model.fit(
    X_full, y_full_team,
    cat_features=categorical_features
)

final_opp_model.fit(
    X_full, y_full_opp,
    cat_features=categorical_features
)


0:	learn: 1.1498593	total: 39.7ms	remaining: 20.6s
100:	learn: 1.0326509	total: 3.6s	remaining: 14.9s
200:	learn: 1.0148399	total: 7.24s	remaining: 11.5s
300:	learn: 0.9996631	total: 11.1s	remaining: 8.11s
400:	learn: 0.9888107	total: 14.9s	remaining: 4.44s
500:	learn: 0.9794131	total: 18.4s	remaining: 736ms
520:	learn: 0.9777443	total: 19.2s	remaining: 0us
0:	learn: 1.1498364	total: 42.6ms	remaining: 29.5s
100:	learn: 1.0344196	total: 3.67s	remaining: 21.5s
200:	learn: 1.0137991	total: 7.33s	remaining: 17.9s
300:	learn: 0.9992809	total: 11s	remaining: 14.3s
400:	learn: 0.9894343	total: 14.7s	remaining: 10.7s
500:	learn: 0.9807268	total: 18.4s	remaining: 7s
600:	learn: 0.9730424	total: 22.1s	remaining: 3.35s
691:	learn: 0.9667692	total: 25.4s	remaining: 0us


In [22]:
test_pred_team = final_team_model.predict(X_test)
test_pred_opp = final_opp_model.predict(X_test)

submission = sample_sub.copy()
submission["team_goals"] = np.round(np.clip(test_pred_team, 0, 5)).astype(int)
submission["opp_goals"] = np.round(np.clip(test_pred_opp, 0, 5)).astype(int)

submission.to_csv("submission_baseline_catboost_fixed.csv", index=False)

print(submission.head())
print("\nSaved: submission_baseline_catboost_fixed.csv")


                   Id  team_goals  opp_goals
0  M034984_Seychelles           1          1
1   M034984_Mauritius           1          1
2     M034985_Comoros           1          1
3    M034985_Maldives           1          1
4     M034986_Réunion           1          1

Saved: submission_baseline_catboost_fixed.csv


In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [24]:
valid_pred_team = team_model.predict(X_valid)
valid_pred_opp = opp_model.predict(X_valid)

print("Raw prediction stats:")
print("team -> min:", np.min(valid_pred_team), "max:", np.max(valid_pred_team), "mean:", np.mean(valid_pred_team))
print("opp  -> min:", np.min(valid_pred_opp), "max:", np.max(valid_pred_opp), "mean:", np.mean(valid_pred_opp))

valid_eval = valid_part.copy()
valid_eval["pred_team_goals"] = np.round(np.clip(valid_pred_team, 0, 5)).astype(int)
valid_eval["pred_opp_goals"] = np.round(np.clip(valid_pred_opp, 0, 5)).astype(int)

# ===== MAE =====
mae_team = mean_absolute_error(valid_eval["team_goals"], valid_eval["pred_team_goals"])
mae_opp = mean_absolute_error(valid_eval["opp_goals"], valid_eval["pred_opp_goals"])
overall_mae = (
    (valid_eval["team_goals"] - valid_eval["pred_team_goals"]).abs() +
    (valid_eval["opp_goals"] - valid_eval["pred_opp_goals"]).abs()
).mean() / 2

# ===== RMSE =====
rmse_team = np.sqrt(mean_squared_error(valid_eval["team_goals"], valid_eval["pred_team_goals"]))
rmse_opp = np.sqrt(mean_squared_error(valid_eval["opp_goals"], valid_eval["pred_opp_goals"]))
overall_rmse = np.sqrt(
    (
        ((valid_eval["team_goals"] - valid_eval["pred_team_goals"]) ** 2) +
        ((valid_eval["opp_goals"] - valid_eval["pred_opp_goals"]) ** 2)
    ).mean() / 2
)

Raw prediction stats:
team -> min: -0.6268733448955486 max: 7.843425647148314 mean: 1.2532732681463352
opp  -> min: -0.5728245052928804 max: 7.812484284872254 mean: 1.2685118009597292


In [25]:
team_denom = np.maximum(np.abs(valid_eval["team_goals"]), 1)
opp_denom = np.maximum(np.abs(valid_eval["opp_goals"]), 1)

mape_team = np.mean(np.abs((valid_eval["team_goals"] - valid_eval["pred_team_goals"]) / team_denom)) * 100
mape_opp = np.mean(np.abs((valid_eval["opp_goals"] - valid_eval["pred_opp_goals"]) / opp_denom)) * 100
overall_mape = (
    np.mean(np.abs((valid_eval["team_goals"] - valid_eval["pred_team_goals"]) / team_denom)) +
    np.mean(np.abs((valid_eval["opp_goals"] - valid_eval["pred_opp_goals"]) / opp_denom))
) / 2 * 100


In [26]:
awmae = aw_mae_score(
    valid_eval,
    true_team_col="team_goals",
    true_opp_col="opp_goals",
    pred_team_col="pred_team_goals",
    pred_opp_col="pred_opp_goals",
    tournament_col="tournament"
)

print("\nValidation MAE team       :", mae_team)
print("Validation MAE opp        :", mae_opp)
print("Validation Overall MAE    :", overall_mae)

print("\nValidation RMSE team      :", rmse_team)
print("Validation RMSE opp       :", rmse_opp)
print("Validation Overall RMSE   :", overall_rmse)

print("\nValidation MAPE team (%)  :", mape_team)
print("Validation MAPE opp (%)   :", mape_opp)
print("Validation Overall MAPE % :", overall_mape)

print("\nValidation AW-MAE         :", awmae)

display(valid_eval[[
    "date", "team", "opponent", "tournament",
    "team_goals", "opp_goals",
    "pred_team_goals", "pred_opp_goals"
]].head(20))


Validation MAE team       : 0.9847188264058679
Validation MAE opp        : 0.9844132029339854
Validation Overall MAE    : 0.9845660146699267

Validation RMSE team      : 1.5155066450524208
Validation RMSE opp       : 1.513488659277099
Validation Overall RMSE   : 1.514497988271763

Validation MAPE team (%)  : 57.14609299185036
Validation MAPE opp (%)   : 57.60484500966411
Validation Overall MAPE % : 57.37546900075723

Validation AW-MAE         : 3.116338852448076


,date,team,opponent,tournament,team_goals,opp_goals,pred_team_goals,pred_opp_goals
37066,2008-10-20,Australia,Vietnam,AFF Championship,1,0,2,1
37067,2008-10-20,Vietnam,Australia,AFF Championship,0,1,1,2
37068,2008-10-20,Myanmar,Thailand,AFF Championship,0,3,1,3
37069,2008-10-20,Afghanistan,Malaysia,Merdeka Tournament,0,6,0,3
37070,2008-10-20,Malaysia,Afghanistan,Merdeka Tournament,6,0,3,0
37071,2008-10-20,Thailand,Myanmar,AFF Championship,3,0,3,1
37072,2008-10-21,Timor-Leste,Brunei,AFF Championship qualification,1,4,2,1
37073,2008-10-21,Brunei,Timor-Leste,AFF Championship qualification,4,1,2,2
37074,2008-10-21,Philippines,Laos,AFF Championship qualification,1,2,2,1
37075,2008-10-21,Laos,Philippines,AFF Championship qualification,2,1,1,2
